# Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.


In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_Key"] = os.getenv("GROQ_API_KEY") 

# Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage

In [24]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("messages", 6),
            keep=("messages", 4)
        )
    ]
)

In [25]:
config = {"configurable":{"thread_id": "1234"}}
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")



Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='1ec60e04-ab41-4499-a7b8-ddd9fc58c3e6'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.009039342, 'completion_tokens_details': None, 'prompt_time': 0.003287532, 'prompt_tokens_details': None, 'queue_time': 0.079691068, 'total_time': 0.012326874}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ece87-2a0a-7db0-be55-baef915dddec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='1ec60e04-ab41-4499-a7b8-ddd9fc58c3e6'), AIMessage

We can also summaries based on message count, token count, fraction

# Human in Loop Middleware

In [26]:
from langchain.tools import tool

@tool
def send_mail(email_id:str):
    """" Send email to provided email """
    return f"Email sent successfully to {email_id}"

@tool
def read_mail(email_id:str):
    """" Read email to provided email """
    return f"Email from {email_id} is \n Hello World"


In [27]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    tools=[send_mail, read_mail],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_mail":{
                    "allowed_decisions": ["accept", "edit", "reject"]
                },
                "read_mail":False
            }
        )
    ]
)

In [28]:
config = {"configurable": {"thread_id":"555"}}

response = agent.invoke({ "messages": [HumanMessage("Send email to john@test.com with subject 'Hello' and body 'How are you?'")]}, config=config)

response

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0cf017b8-b4c3-41bc-aa3f-1c42d2163f0e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hbg5drn96', 'function': {'arguments': '{"body":"How are you?","email_id":"john@test.com","subject":"Hello"}', 'name': 'send_mail'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 283, 'total_tokens': 315, 'completion_time': 0.028830912, 'completion_tokens_details': None, 'prompt_time': 0.018164353, 'prompt_tokens_details': None, 'queue_time': 0.049089657, 'total_time': 0.046995265}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ece87-39ac-7af1-b299-c19683f1e782-0', tool_calls=[{'name': 'send_mail', 'args': {'body': 'How are y

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in response:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...


ValueError: Unexpected human decision: {'type': 'approve'}. Decision type 'approve' is not allowed for tool 'send_mail'. Expected one of ['accept', 'edit', 'reject'] based on the tool's configuration.

# Custom Middleware